In [1]:
!python data.py

In [2]:
# Data Cleaning
import pandas as pd

df = pd.read_csv("house_sales.csv")
missing_city = int((df['city'] == '--').sum())

clean_data = df.copy()
clean_data = clean_data.dropna(subset=['sale_price'])
clean_data['city'] = clean_data['city'].replace('--', 'Unknown').fillna('Unknown')
clean_data['sale_date'] = clean_data['sale_date'].fillna("2023-01-01")
clean_data['months_listed'] = clean_data['months_listed'].fillna(round(clean_data['months_listed'].mean(), 1))
clean_data['bedrooms'] = clean_data['bedrooms'].fillna(round(clean_data['bedrooms'].mean()))

mapping = {'Det.': 'Detached', 'Semi': 'Semi-detached', 'Terr.': 'Terraced'}
clean_data['house_type'] = clean_data['house_type'].replace(mapping).fillna(clean_data['house_type'].mode()[0])

clean_data['area'] = clean_data['area'].str.replace(' sq.m.', '', regex=False).astype(float)
clean_data['area'] = clean_data['area'].fillna(round(clean_data['area'].mean(), 1))
clean_data

In [3]:
# EDA
price_by_rooms = clean_data.groupby('bedrooms')['sale_price'].agg(['mean', 'var']).reset_index()
price_by_rooms.columns = ['bedrooms', 'avg_price', 'var_price']
price_by_rooms = price_by_rooms.round(1)
price_by_rooms

In [4]:
# Data Preprocessing
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

train = pd.read_csv('train.csv')
validation = pd.read_csv('validation.csv')

mapping = {'Det.': 'Detached', 'Semi': 'Semi-detached', 'Terr.': 'Terraced'}
for d in [train, validation]:
    d['city'] = d['city'].replace('--', 'Unknown').fillna('Unknown')
    d['house_type'] = d['house_type'].replace(mapping).fillna('Detached')
    d['area'] = d['area'].fillna(d['area'].mean())
    d['bedrooms'] = d['bedrooms'].fillna(d['bedrooms'].mean())
    d['months_listed'] = d['months_listed'].fillna(d['months_listed'].mean())

train = train.dropna(subset=['sale_price'])

features = ['area', 'bedrooms', 'months_listed', 'city', 'house_type']
X_train = pd.get_dummies(train[features], drop_first=True)
X_val = pd.get_dummies(validation[features], drop_first=True)

X_train, X_val = X_train.align(X_val, join='left', axis=1, fill_value=0)
y_train = train['sale_price']

In [5]:
# Model Building
base_model = LinearRegression()
base_model.fit(X_train, y_train)

base_result = pd.DataFrame({
    'house_id': validation['house_id'],
    'price': base_model.predict(X_val)
})

compare_model = RandomForestRegressor(n_estimators=100, random_state=42)
compare_model.fit(X_train, y_train)

compare_result = pd.DataFrame({
    'house_id': validation['house_id'],
    'price': compare_model.predict(X_val)
})